In [ ]:
from pathlib import Path

import equinox as eqx
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import optax
import xarray as xr
from context_flux_no.models.multiphysics import (
    AbstractMultiphysicsOperator,
    DISCO,
    DPOT,
)
from context_flux_no.models.multiphysics.hyperfluxfno import (
    HyperFluxFNOLocal,
)
from context_flux_no.training.loader import (
    SegmentLoaderBackground,
)
from context_flux_no.training.trainer import Trainer
from jaxtyping import Array, Float, PRNGKeyArray


jax.config.update("jax_default_device", jax.devices("gpu")[5])

In [ ]:
datadir = Path("../../data/datasets/shallow_water_1d/data")
dataset_xr = xr.open_dataset(
    datadir / "valid" / "shallow_water_1d_validation_grf_seed=11.hdf5",
    engine="h5netcdf",
    chunks={},
)  # .isel({"t": slice(0, 80)})
dataset_xr

In [ ]:
datadir = Path("../../data/datasets/shallow_water_1d/data")
dataset_xr = xr.open_dataset(
    datadir / "train/orig" / "shallow_water_1d_train_grf_seed=8.hdf5",
    engine="h5netcdf",
    chunks={},
)  # .isel({"t": slice(0, 80)})
dataset_xr

dataset_xr = dataset_xr.sel({"ic": ~np.asarray(has_nan)})
dataset_xr.to_netcdf(datadir / "train" / "shallow_water_1d_train_grf_seed=8.hdf5")

In [ ]:
val = dataset_xr["values"].values
has_nan = jax.vmap(jnp.any)(jnp.isnan(val))
jnp.sum(has_nan)

In [ ]:
jnp.argwhere(has_nan)

In [ ]:
sample_idx = 0
fig, axes = plt.subplots(2, 5, figsize=(8, 3), sharey="row")
for i, axes_row in enumerate(axes):
    for j, ax in enumerate(axes_row):
        ax.plot(dataset_xr.coords["x"], dataset_xr["values"][sample_idx, 10 * j, i])

In [ ]:
datadir = Path("../../data/datasets/shallow_water_1d/data/train")
dataset_xr = xr.open_mfdataset(
    sorted(list(datadir.glob("shallow_water_1d_train_grf_seed=*.hdf5"))),
    combine="nested",
    concat_dim="ic",
    engine="h5netcdf",
)

In [ ]:
dataset_xr

In [ ]:
dataloader = SegmentLoaderBackground(
    dataset_xr, segment_length=21, batch_size=256, batches_per_load=100
)

In [ ]:
dataiter = iter(dataloader)

In [ ]:
for i in range(520):
    arr = next(dataiter)
    print(i, jnp.linalg.norm(jnp.reshape(arr, shape=(-1,))))

In [ ]:
for j, a in enumerate(arr):
    print(j, jnp.any(jnp.isnan(a)))

In [ ]:
sample_idx = 116
fig, axes = plt.subplots(2, 5, figsize=(8, 3), sharey="row", sharex=True)
for i, axes_row in enumerate(axes):
    for j, ax in enumerate(axes_row):
        ax.plot(dataset_xr.coords["x"], arr[sample_idx, 10 * j, i])

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(5, 3))

In [ ]:
arr = next(dataiter)

In [ ]:
arr.dtype.itemsize * arr.size * 50 / 1e9

In [ ]:
arr.shape

In [ ]:
import numpy as np


np.all(np.isnan(arr))

In [ ]:
sample_idx = 60
fig, axes = plt.subplots(2, 5, figsize=(8, 4), sharex=True, sharey="row")
for i, ax_row in enumerate(axes):
    for j, ax in enumerate(ax_row):
        ax.plot(arr[sample_idx, 4 * j, i])

In [ ]:
dpot = DPOT(
    num_spatial_dims=1,
    in_channels=2,
    out_channels=2,
    in_timesteps=20,
    grid_size=(100,),
    patch_size=(4,),
    embedding_dim=128,
    max_frequency_modes=12,
    fno_depth=10,
    num_blocks=8,
    hidden_dim_patch=512,
    hidden_dim_fno=512,
    hidden_dim_output=512,
    key=jax.random.key(0),
)
print(dpot.num_parameters() / 1e6)

In [ ]:
hyperfluxfno = HyperFluxFNOLocal(
    num_spatial_dims=1,
    in_channels=2,
    in_timesteps=20,
    embedding_dim=128,
    encoder_type="TRecViT",
    encoder_kwargs=dict(
        grid_size=(100,),
        patch_size=(4,),
        depth=2,
        temporal_block_width=128,
        num_heads=8,
        mlp_hidden_dim=64,
    ),
    depth=4,
    lift_dim=128,
    stencil_size=(10, 10),
    width_hyper=128,
    blocks_hyper=4,
    key=jax.random.key(0),
)
print(hyperfluxfno.num_parameters() / 1e6)

In [ ]:
disco = DISCO(
    num_spatial_dims=1,
    channels=2,
    embedding_dim=128,
    patch_size=4,
    num_hypernet_blocks=3,
    droppath=0.1,
    num_hypernet_heads=8,
    mlp_hidden_dim=32,
    boundary_condition="periodic",
    opnet_hidden_channels=4,
    opnet_norm_groups=2,
    rtol=1e-4,
    atol=1e-6,
    max_steps=16,
    key=jax.random.key(0),
)
print(disco.num_parameters() / 1e6)

In [ ]:
def loss_fn(
    model: AbstractMultiphysicsOperator,
    u: Float[Array, "batch time dim ..."],
    args,
    key: PRNGKeyArray,
) -> tuple[Float[Array, ""], dict]:
    u0, u1 = u[:, :-1], u[:, -1]
    keys = jax.random.split(key, u0.shape[0])
    u1_pred: Float[Array, "batch dim ..."] = eqx.filter_vmap(
        lambda u_, key_: model(u_, args, key=key_)
    )(u0, keys)[0]
    return jnp.mean((u1 - u1_pred) ** 2), dict()


trainer = Trainer(
    optax.adamw(1e-3),
    loss_fn,
    "./",
    None,
    {"entity": "jhko725", "project": "hyperfluxfno"},
)